In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score
import xgboost as xgb
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
import mlflow
import mlflow.sklearn

# 데이터 로딩
data = pd.read_csv('../data/dataset.csv')

In [2]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")

In [3]:
# 예측 타겟과 특성 분리
X = data.drop('Default', axis=1)
y = data['Default']

In [4]:
# 범주형과 수치형 컬럼 구분
categorical_cols = ['State', 'BankState', 'NewExist', 'UrbanRural', 'RealEstate']
numerical_cols = ['DisbursementGross', 'GrAppv', 'daysterm']


In [5]:
# 데이터 전처리 파이프라인 구성
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

In [6]:
# 데이터 전처리 실행
X_processed = preprocessor.fit_transform(X)

In [7]:
# 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42)

# 하이퍼파라미터 탐색 공간 설정
space = {
    'max_depth': hp.choice('max_depth', range(3, 10)),
    'learning_rate': hp.uniform('learning_rate', 0.01, 0.2),
    'n_estimators': hp.choice('n_estimators', range(50, 200)),
    'gamma': hp.uniform('gamma', 0, 5)
}


In [8]:
# 최적화를 위한 목적 함수 정의
def objective(params):
    # MLflow에 실험 이름 설정
    mlflow.set_experiment("assignment1")

    # 각 하이퍼파라미터 조합별 실험 시작
    with mlflow.start_run(nested=True):
        # XGBoost 모델 초기화 및 훈련
        model = xgb.XGBClassifier(eval_metric='logloss', use_label_encoder=False, **params)
        model.fit(X_train, y_train)

        # 예측 확률 및 ROC-AUC 계산
        probs = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, probs)

        # MLflow에 파라미터 및 메트릭 로깅
        mlflow.log_params(params)
        mlflow.log_metric("roc_auc", auc)

        # 목적 함수 결과 반환 (손실 최소화)
        return {'loss': -auc, 'status': STATUS_OK}


In [9]:
# Hyperopt를 사용한 최적화 실행
trials = Trials()
best_params = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=30,
    trials=trials
)

  0%|          | 0/30 [00:00<?, ?trial/s, best loss=?]

2026/04/22 12:07:20 INFO mlflow.tracking.fluent: Experiment with name 'assignment1' does not exist. Creating a new experiment.



🏃 View run able-mouse-174 at: http://127.0.0.1:5000/#/experiments/12/runs/7a1054fcb04b464c81ff624d722d7088

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/12

  3%|▎         | 1/30 [00:00<00:13,  2.14trial/s, best loss: -0.9716612216612216]

c:\Users\SSAFY\Desktop\TIL\09_data_analysis\g_mlflow\mlflow_env\Lib\site-packages\xgboost\training.py:183: UserWarning: [12:07:21] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)

c:\Users\SSAFY\Desktop\TIL\09_data_analysis\g_mlflow\mlflow_env\Lib\site-packages\xgboost\training.py:183: UserWarning: [12:07:21] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



🏃 View run dashing-worm-377 at: http://127.0.0.1:5000/#/experiments/12/runs/02cf77263fec45bb964b02edc057723d

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/12                    

🏃 View run tasteful-colt-425 at: http://127.0.0.1:5000/#/experiments/12/runs/f075532e7563414eb72fed4f7dc2ded1

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/12                    

 10%|█         | 3/30 [00:00<00:05,  4.53trial/s, best loss: -0.9757821007821008]

...

🏃 View run sincere-mole-22 at: http://127.0.0.1:5000/#/experiments/12/runs/2670a3603da84b17aec66ac1dc149cf2

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/12                     

🏃 View run bemused-worm-323 at: http://127.0.0.1:5000/#/experiments/12/runs/b86addd42eb14f319c8db0713a91dfd2

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/12                     

 90%|█████████ | 27/30 [00:04<00:00,  6.41trial/s, best loss: -0.9794698544698545]

c:\Users\SSAFY\Desktop\TIL\09_data_analysis\g_mlflow\mlflow_env\Lib\site-packages\xgboost\training.py:183: UserWarning: [12:07:25] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)

c:\Users\SSAFY\Desktop\TIL\09_data_analysis\g_mlflow\mlflow_env\Lib\site-packages\xgboost\training.py:183: UserWarning: [12:07:25] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



🏃 View run nebulous-ant-558 at: http://127.0.0.1:5000/#/experiments/12/runs/4b8574d9ac6341b991a0e22ed51f5df7

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/12                     

🏃 View run learned-hog-219 at: http://127.0.0.1:5000/#/experiments/12/runs/ec2d26aa734a427b84398d2d56e32ad2

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/12                     

 97%|█████████▋| 29/30 [00:04<00:00,  6.43trial/s, best loss: -0.9794698544698545]

c:\Users\SSAFY\Desktop\TIL\09_data_analysis\g_mlflow\mlflow_env\Lib\site-packages\xgboost\training.py:183: UserWarning: [12:07:25] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)

c:\Users\SSAFY\Desktop\TIL\09_data_analysis\g_mlflow\mlflow_env\Lib\site-packages\xgboost\training.py:183: UserWarning: [12:07:25] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



🏃 View run gaudy-hound-877 at: http://127.0.0.1:5000/#/experiments/12/runs/c9e3f414838a47c4835e8de9431547c4

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/12                     

100%|██████████| 30/30 [00:04<00:00,  6.11trial/s, best loss: -0.9794698544698545]
